# 03. Data Pipeline Integration

This notebook demonstrates the complete data pipeline with all components integrated.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import our modules
from src.data.data_loader import PayloadByteDataLoader, create_pytorch_dataloaders
from src.data.cache_manager import CacheManager
from src.data.data_versioning import DataVersionManager
from src.utils.config_manager import get_config_manager
from src.utils.data_validation import DataValidator
from src.utils.logger import setup_logger

logger = setup_logger('data_pipeline_demo')

## 1. Configuration Management

In [ ]:
# Load configuration
config_mgr = get_config_manager()

# Display data configuration
data_config = config_mgr.get_data_config()
print("Data Configuration:")
print(f"  Raw data path: {data_config['data']['raw_data_path']}")
print(f"  Chunk size: {data_config['data']['chunk_size']}")
print(f"  Workers: {data_config['data']['n_workers']}")
print(f"\nSplit ratios:")
for split, ratio in data_config['data']['split_ratios'].items():
    print(f"  {split}: {ratio}")

## 2. Data Validation

In [ ]:
# Validate datasets
validator = DataValidator()
data_path = project_root / data_config['data']['raw_data_path']

# Run validation
validation_results = validator.validate_dataset(data_path)
validator.print_summary(validation_results)

## 3. Data Versioning

In [ ]:
# Initialize version manager
version_mgr = DataVersionManager()

# Create version for our datasets
for csv_file in data_path.glob('*.csv'):
    if 'sample' in csv_file.name:
        version = version_mgr.create_version(
            data_path=csv_file,
            description=f"Pipeline demo version of {csv_file.name}",
            tags=['demo', 'pipeline']
        )
        print(f"Created version: {version.version_id} for {csv_file.name}")

# List all versions
print("\nAll data versions:")
for v in version_mgr.list_versions():
    print(f"  {v.version_id}: {v.metadata['description']}")

## 4. Data Loading with Caching

In [ ]:
# Initialize cache manager
cache_mgr = CacheManager()
print(f"Cache enabled: {cache_mgr.enabled}")
print(f"Cache format: {cache_mgr.format}")
print(f"Cache TTL: {cache_mgr.ttl_hours} hours")

# Initialize data loader
loader = PayloadByteDataLoader(
    data_path=data_path,
    batch_size=config_mgr.get('data_config', 'pipeline.batch_size', 32),
    chunk_size=data_config['data']['chunk_size'],
    n_workers=data_config['data']['n_workers']
)

In [ ]:
# Load data with caching
import time

# First load (will be cached)
start_time = time.time()
data = loader.load_raw_packets(nrows=1000)  # Load subset for demo
first_load_time = time.time() - start_time
print(f"First load time: {first_load_time:.2f}s")
print(f"Loaded {len(data)} rows")

# Cache the data
cache_key_path = list(data_path.glob('*.csv'))[0]
cache_mgr.save_to_cache(data, cache_key_path, {'nrows': 1000})

# Second load (from cache)
start_time = time.time()
cached_data = cache_mgr.load_from_cache(cache_key_path, {'nrows': 1000})
cache_load_time = time.time() - start_time

if cached_data is not None:
    print(f"\nCache load time: {cache_load_time:.2f}s")
    print(f"Speedup: {first_load_time / cache_load_time:.1f}x")
else:
    print("Cache miss")

## 5. Train/Validation/Test Splitting

In [ ]:
# Create splits
train_df, val_df, test_df = loader.create_train_val_test_splits(
    data=data,
    train_ratio=data_config['data']['split_ratios']['train'],
    val_ratio=data_config['data']['split_ratios']['validation'],
    test_ratio=data_config['data']['split_ratios']['test'],
    stratify=data_config['data']['stratify'],
    random_state=data_config['data']['random_state']
)

# Display split statistics
stats = loader.get_data_stats()
print("\nData Split Statistics:")
print(f"  Train samples: {stats['n_train']}")
print(f"  Val samples: {stats['n_val']}")
print(f"  Test samples: {stats['n_test']}")

# Check label distribution
if 'train_labels' in stats:
    print("\nTrain set label distribution:")
    for label, count in stats['train_labels'].items():
        print(f"  Label {label}: {count} samples")

## 6. Batch Generation

In [ ]:
# Test batch generator
print("Testing batch generator:")
batch_gen = loader.get_batch_generator('train', shuffle=True)

for i, batch in enumerate(batch_gen):
    print(f"  Batch {i}: shape={batch.shape}")
    if i >= 4:  # Show first 5 batches
        break

# Visualize a batch
first_batch = next(loader.get_batch_generator('train'))
print(f"\nFirst batch details:")
print(f"  Shape: {first_batch.shape}")
print(f"  Columns: {len(first_batch.columns)}")
print(f"  Memory usage: {first_batch.memory_usage().sum() / 1024:.2f} KB")

## 7. PyTorch Integration

In [ ]:
# Create PyTorch DataLoaders
dataloaders = create_pytorch_dataloaders(
    loader,
    batch_size=32,
    num_workers=2,
    pin_memory=False  # Set to True if using GPU
)

print("PyTorch DataLoaders created:")
for name, dataloader in dataloaders.items():
    print(f"  {name}: {len(dataloader)} batches")

# Test PyTorch DataLoader
train_loader = dataloaders['train']
batch_x, batch_y = next(iter(train_loader))
print(f"\nPyTorch batch:")
print(f"  Features shape: {batch_x.shape}")
print(f"  Labels shape: {batch_y.shape}")
print(f"  Features dtype: {batch_x.dtype}")
print(f"  Labels dtype: {batch_y.dtype}")

## 8. Cache Statistics

In [ ]:
# Display cache statistics
cache_stats = cache_mgr.get_cache_stats()
print("\nCache Statistics:")
for key, value in cache_stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.2f}")
    else:
        print(f"  {key}: {value}")

## 9. Pipeline Performance Metrics

In [ ]:
# Measure pipeline performance
import time

performance_metrics = {}

# Data loading speed
start = time.time()
_ = loader.load_raw_packets(nrows=100)
performance_metrics['load_time_per_100_rows'] = time.time() - start

# Batch generation speed
start = time.time()
batches = list(loader.get_batch_generator('train'))
performance_metrics['batch_generation_time'] = time.time() - start
performance_metrics['batches_per_second'] = len(batches) / performance_metrics['batch_generation_time']

# Memory usage
import psutil
process = psutil.Process()
performance_metrics['memory_usage_mb'] = process.memory_info().rss / 1024 / 1024

print("Pipeline Performance Metrics:")
for metric, value in performance_metrics.items():
    print(f"  {metric}: {value:.2f}")

## Summary

The data pipeline successfully integrates:

1. **Configuration Management**: Centralized YAML-based configuration
2. **Data Validation**: Comprehensive validation of data format and integrity
3. **Data Versioning**: Track dataset versions and lineage
4. **Efficient Loading**: Chunked reading with multi-threading support
5. **Caching**: HDF5-based caching for fast data access
6. **Train/Val/Test Splitting**: Stratified splitting with configurable ratios
7. **Batch Generation**: Memory-efficient batch generation
8. **PyTorch Integration**: Ready-to-use DataLoaders for model training

The pipeline is modular, configurable, and optimized for performance.